# 📝 [제 10편] 신경망과 딥러닝 - 피드포워드로 XOR 풀기

**학년/반/번호:** 2학년 ( )반 ( )번  **이름:** (           )

> 근거 자료: 박성호, 「Deep Learning (I) 신경망 개념 • 아키텍처」, 「(II) feed forward • loss」, 「(III) 딥러닝으로 XOR 문제 해결」 (ML_LEC_19, 20, 21)

---

## 🎯 학습 목표
1. 뉴런과 활성화 함수의 관계를 알고, 입력층·은닉층·출력층으로 된 신경망 구조를 설명할 수 있다.
2. 가중치·바이어스 표기법을 읽고, 피드포워드(순전파) 계산을 일반 식과 행렬 식으로 할 수 있다.
3. 은닉층이 있는 신경망을 학습시켜 XOR 문제를 해결할 수 있다.

---

## 🧭 지난 시간 복습
* 제 9편: 로지스틱 회귀 하나로는 XOR 을 풀 수 없지만, NAND·OR (1층) → AND (2층) 처럼 **층을 쌓으면** 풀린다.
* 오늘은 층 사이의 가중치를 사람이 설계하지 않고 **신경망이 스스로 학습** 하게 합니다.

[⬅ 이전 : 제 9편 XOR 문제 - 직선 하나로는 나눌 수 없다](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/09_xor.ipynb)　|　[📚 목차](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/00_index.ipynb)　|　[다음 : 제 11편 MNIST - 손글씨 숫자를 읽는 신경망 ➡](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/11_mnist.ipynb)

> 💡 **처음 여는 경우** : 파일 → **Drive에 사본 저장** 을 누르고, 내 사본에서 공부하세요. (빈칸을 채우고 저장할 수 있습니다)
>
> 🧹 **다 본 뒤에는** : 수정 → **모든 출력 지우기** 후 저장하면 애니메이션 때문에 무거워진 사본이 다시 가벼워집니다.

## ⚙️ 0단계. 준비하기
아래 셀을 **가장 먼저 한 번** 실행하세요. (그래프 한글 글꼴과 애니메이션 설정)

In [ ]:
#@title ▶ 준비 셀 (먼저 실행하세요)
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib import animation
from IPython.display import HTML, display
import urllib.request, os

font_path = 'NanumGothic.ttf'
try:
    if not os.path.exists(font_path):
        urllib.request.urlretrieve('https://raw.githubusercontent.com/google/fonts/main/ofl/nanumgothic/NanumGothic-Regular.ttf', font_path)
    fm.fontManager.addfont(font_path)
    plt.rc('font', family=fm.FontProperties(fname=font_path).get_name())
except Exception as e:
    print('글꼴을 내려받지 못했습니다(그래프의 한글이 깨질 수 있음):', e)
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['animation.embed_limit'] = 50
def show_anim(anim):
    # 애니메이션을 코랩 화면에 재생 버튼과 함께 보여 주기
    display(HTML(anim.to_jshtml()))
np.set_printoptions(precision=6, suppress=True)
print('✅ 준비 완료!')

In [ ]:
#@title ▶ 수치미분, sigmoid 함수 (실행만 하세요)
def numerical_derivative(f, x):
    delta_x = 1e-4
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])
    while not it.finished:
        idx = it.multi_index
        tmp_val = x[idx]
        x[idx] = float(tmp_val) + delta_x; fx1 = f(x)
        x[idx] = float(tmp_val) - delta_x; fx2 = f(x)
        grad[idx] = (fx1 - fx2) / (2 * delta_x)
        x[idx] = tmp_val
        it.iternext()
    return grad

def sigmoid(z):
    return 1 / (1 + np.exp(-z))
print('준비 완료')

## 📖 개념 다지기 1. 뉴런과 활성화 함수

우리 뇌의 신경세포(뉴런)는
1. 앞 뉴런들에게서 신호를 받아 (입력 × 가중치 를 모두 더하고)
2. 그 합이 어느 **임계점** 을 넘을 때만 다음 뉴런에 신호를 보냅니다.

이처럼 입력의 합이 임계점을 넘으면 출력을 만들어 주는 함수를 **활성화 함수 (activation function)** 라고 합니다.
지금까지 써 온 **sigmoid** 가 대표적인 활성화 함수입니다. (그 밖에 ReLU, tanh 등)

> **인공 뉴런 (노드) 1개 = 로지스틱 회귀 1개**
> z = x₁w₁ + x₂w₂ + … + b  (선형 회귀) 　→　 a = sigmoid(z)  (활성화)

## 📖 개념 다지기 2. 신경망의 구조

| 층 | 하는 일 |
|:---|:---|
| **입력층 (input layer)** | 데이터를 그대로 받아 다음 층으로 넘긴다 (계산 없음, sigmoid 도 적용하지 않음) |
| **은닉층 (hidden layer)** | 중간 계산. 1개 이상, 노드 개수는 마음대로 정할 수 있다 |
| **출력층 (output layer)** | 최종 결과 y 를 낸다 |

* 은닉층이 1개 이상 있는 신경망을 학습시키는 것을 **딥러닝(Deep Learning)** 이라고 합니다. 은닉층이 깊을수록(deep) 복잡한 문제를 풀 수 있습니다.

## 📖 개념 다지기 3. 표기법 (notation)

층 번호는 입력층 = 1, 은닉층 = 2, 출력층 = 3 으로 셉니다.

| 기호 | 뜻 |
|:---|:---|
| w<sub>21</sub><sup>(2)</sup> | **계층 2** 에 적용되는 가중치. 계층 1 의 **노드 1** 에서 계층 2 의 **노드 2** 로 가는 신호에 곱한다 (다음 층 노드 번호가 **먼저**!) |
| b<sub>1</sub><sup>(2)</sup> | 계층 2 의 노드 1 에 더하는 바이어스 |
| z<sub>2</sub><sup>(2)</sup> | 계층 2 의 노드 2 의 선형 회귀 값 |
| a<sub>2</sub><sup>(2)</sup> | 계층 2 의 노드 2 의 출력 = sigmoid(z<sub>2</sub><sup>(2)</sup>) |

> **[연습 1]** w<sub>13</sub><sup>(3)</sup> 은 계층 ( 　 ) 의 노드 ( 　 ) 에서 계층 ( 　 ) 의 노드 ( 　 ) 로 가는 신호의 가중치이다.

## 📖 개념 다지기 4. 피드포워드 (feed forward, 순전파)

입력층 → 은닉층 → 출력층 으로 값이 **앞으로 전달** 되며 계산되는 과정입니다. (입력 2개, 은닉 노드 2개, 출력 1개)

**① 입력층 출력**:  a<sub>1</sub><sup>(1)</sup> = x₁,  a<sub>2</sub><sup>(1)</sup> = x₂

**② 은닉층**
* z<sub>1</sub><sup>(2)</sup> = a<sub>1</sub><sup>(1)</sup> w<sub>11</sub><sup>(2)</sup> + a<sub>2</sub><sup>(1)</sup> w<sub>12</sub><sup>(2)</sup> + b<sub>1</sub><sup>(2)</sup>,　 a<sub>1</sub><sup>(2)</sup> = sigmoid(z<sub>1</sub><sup>(2)</sup>)
* z<sub>2</sub><sup>(2)</sup> = a<sub>1</sub><sup>(1)</sup> w<sub>21</sub><sup>(2)</sup> + a<sub>2</sub><sup>(1)</sup> w<sub>22</sub><sup>(2)</sup> + b<sub>2</sub><sup>(2)</sup>,　 a<sub>2</sub><sup>(2)</sup> = sigmoid(z<sub>2</sub><sup>(2)</sup>)

**③ 출력층**
* z<sub>1</sub><sup>(3)</sup> = a<sub>1</sub><sup>(2)</sup> w<sub>11</sub><sup>(3)</sup> + a<sub>2</sub><sup>(2)</sup> w<sub>12</sub><sup>(3)</sup> + b<sub>1</sub><sup>(3)</sup>,　 y = a<sub>1</sub><sup>(3)</sup> = sigmoid(z<sub>1</sub><sup>(3)</sup>)

### 행렬 식으로 쓰면 한 줄!
> **Z<sup>(2)</sup> = A<sup>(1)</sup> · W<sup>(2)</sup> + b<sup>(2)</sup>,　A<sup>(2)</sup> = sigmoid(Z<sup>(2)</sup>)**
> **Z<sup>(3)</sup> = A<sup>(2)</sup> · W<sup>(3)</sup> + b<sup>(3)</sup>,　y = A<sup>(3)</sup> = sigmoid(Z<sup>(3)</sup>)**

(1 × 2) · (2 × 2) = (1 × 2),　(1 × 2) · (2 × 1) = (1 × 1)

### [연습 2] 손으로 피드포워드 계산하기

아래 가중치는 **제 9편의 OR, NAND, AND** 를 흉내 내도록 골랐습니다. (sigmoid(10) ≒ 1, sigmoid(−10) ≒ 0 으로 계산하세요)

* 은닉 노드 1 (OR 역할):  z<sub>1</sub><sup>(2)</sup> = 20x₁ + 20x₂ − 10
* 은닉 노드 2 (NAND 역할): z<sub>2</sub><sup>(2)</sup> = −20x₁ − 20x₂ + 30
* 출력 노드 (AND 역할):  z<sup>(3)</sup> = 20a<sub>1</sub><sup>(2)</sup> + 20a<sub>2</sub><sup>(2)</sup> − 30

| (x₁, x₂) | z<sub>1</sub><sup>(2)</sup> → a<sub>1</sub><sup>(2)</sup> | z<sub>2</sub><sup>(2)</sup> → a<sub>2</sub><sup>(2)</sup> | z<sup>(3)</sup> → y |
|:-:|:-:|:-:|:-:|
| (0, 0) | −10 → 0 | 30 → 1 | 20·0 + 20·1 − 30 = −10 → 0 |
| (0, 1) | 10 → ( 　 ) | 10 → ( 　 ) | ( 　　 ) → ( 　 ) |
| (1, 0) | ( 　 ) → 1 | ( 　 ) → 1 | 10 → 1 |
| (1, 1) | 30 → 1 | ( 　 ) → ( 　 ) | ( 　　 ) → ( 　 ) |

> 결과 y 는 ( AND / OR / XOR ) 의 진리표와 같다!

In [ ]:
W2 = np.array([[20, -20],
               [20, -20]])          # 입력층 → 은닉층 가중치 (2 × 2) : 열 1 = OR 노드, 열 2 = NAND 노드
b2 = np.array([-10, 30])
W3 = np.array([[20], [20]])         # 은닉층 → 출력층 가중치 (2 × 1)
b3 = np.array([-30])

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
for x in X:
    A1 = x                                   # 입력층 출력
    Z2 = np.dot(A1, W2) + b2; A2 = sigmoid(Z2)   # 은닉층
    Z3 = np.dot(A2, W3) + b3; A3 = sigmoid(Z3)   # 출력층
    print(f'{x} → Z2 = {Z2}, A2 = {A2.round(3)} → Z3 = {Z3.round(2)}, y = {A3.round(4)}')

## 🎬 애니메이션 1. 신호가 앞으로 흘러가는 피드포워드

위 신경망에 입력 (0,0), (0,1), (1,0), (1,1) 을 차례로 넣습니다. 노드 안의 숫자는 그 노드의 출력 a, 선의 굵기는 가중치의 크기(빨강 +, 파랑 −) 입니다.

In [ ]:
#@title 🎬 애니메이션 1 실행 (피드포워드)
pos = {('in', 0): (0, 2), ('in', 1): (0, 0), ('h', 0): (2, 2), ('h', 1): (2, 0), ('out', 0): (4, 1)}
edges = [(('in', i), ('h', j), W2[i, j]) for i in range(2) for j in range(2)] + [(('h', i), ('out', 0), W3[i, 0]) for i in range(2)]
seq = []
for x in X:
    for phase in range(4):
        seq += [(tuple(int(v) for v in x), phase)] * 5
fig, ax = plt.subplots(figsize=(9, 5))
def draw(f):
    x, phase = seq[f]
    A1 = np.array(x, float); A2 = sigmoid(A1 @ W2 + b2); A3 = sigmoid(A2 @ W3 + b3)
    vals = {('in', 0): A1[0], ('in', 1): A1[1], ('h', 0): A2[0], ('h', 1): A2[1], ('out', 0): A3[0]}
    shown = {'in': phase >= 0, 'h': phase >= 2, 'out': phase >= 3}
    ax.clear(); ax.set_xlim(-1, 5.6); ax.set_ylim(-0.9, 2.9); ax.axis('off')
    for (p, q, w) in edges:
        active = (p[0] == 'in' and phase >= 1) or (p[0] == 'h' and phase >= 3)
        ax.plot([pos[p][0], pos[q][0]], [pos[p][1], pos[q][1]], color='crimson' if w > 0 else 'royalblue',
                lw=abs(w)/4, alpha=0.9 if active else 0.15)
    for k, (cx, cy) in pos.items():
        v = vals[k] if shown[k[0]] else None
        fc = plt.cm.Greens(0.2 + 0.7*v) if v is not None else 'white'
        ax.add_patch(plt.Circle((cx, cy), 0.42, fc=fc, ec='k', lw=2, zorder=3))
        ax.text(cx, cy, '' if v is None else f'{v:.2f}', ha='center', va='center', fontsize=13, zorder=4, weight='bold')
    ax.text(0, 2.65, '입력층', ha='center', fontsize=12); ax.text(2, 2.65, '은닉층 (OR, NAND)', ha='center', fontsize=12)
    ax.text(4, 1.65, '출력층 (AND)', ha='center', fontsize=12)
    if phase >= 3: ax.text(4.75, 1, f'y = {int(A3[0] > 0.5)}', fontsize=16, va='center', color='darkgreen', weight='bold')
    ax.set_title(f'입력 (x1, x2) = {x}', fontsize=15)
anim = animation.FuncAnimation(fig, draw, frames=len(seq), interval=160)
plt.close(fig)
show_anim(anim)

## 💻 실습. 딥러닝으로 XOR 학습시키기 (2 - 6 - 1 신경망)

이번에는 가중치를 정해 주지 않고 **무작위 값에서 시작해 스스로 학습** 시킵니다.
* 입력층 노드 2개, **은닉층 노드 6개**, 출력층 노드 1개
* W2 : (2 × 6),  b2 : 6개,  W3 : (6 × 1),  b3 : 1개 → 학습할 값이 모두 ( 　 ) 개
* 학습: W2, b2, W3, b3 를 **모두** 경사하강법으로 갱신

```
 ① 입력 → ② feed forward → ③ 출력 y 와 정답 t 로 손실 계산 → ④ W2, b2, W3, b3 갱신 → ① 반복
```

In [ ]:
class LogicGate:
    def __init__(self, gate_name, xdata, tdata):
        self.name = gate_name
        self.__xdata = xdata.reshape(4, 2)
        self.__tdata = tdata.reshape(4, 1)
        # 은닉층 (노드 6개)
        self.__W2 = np.random.rand(2, 6)
        self.__b2 = np.random.rand(6)
        # 출력층 (노드 1개)
        self.__W3 = np.random.rand(6, 1)
        self.__b3 = np.random.rand(1)
        self.__learning_rate = 1e-2
        self.history = []

    def feed_forward(self):                          # 피드포워드로 손실 계산
        delta = 1e-7
        z2 = np.dot(self.__xdata, self.__W2) + self.__b2
        a2 = sigmoid(z2)
        z3 = np.dot(a2, self.__W3) + self.__b3
        y = sigmoid(z3)
        return -np.sum(self.__tdata*np.log(y + delta) + (1 - self.__tdata)*np.log((1 - y) + delta))

    def loss_val(self):
        return self.feed_forward()

    def train(self, steps=10001):
        f = lambda x : self.feed_forward()
        print('처음 손실 =', round(self.loss_val(), 5))
        for step in range(steps):
            self.__W2 -= self.__learning_rate * numerical_derivative(f, self.__W2)
            self.__b2 -= self.__learning_rate * numerical_derivative(f, self.__b2)
            self.__W3 -= self.__learning_rate * numerical_derivative(f, self.__W3)
            self.__b3 -= self.__learning_rate * numerical_derivative(f, self.__b3)
            if step % 100 == 0:
                self.history.append((step, self.__W2.copy(), self.__b2.copy(), self.__W3.copy(), self.__b3.copy()))
            if step % 2000 == 0:
                print(f'step {step:5d} : 손실 = {self.loss_val():.5f}')

    def predict(self, xdata):
        z2 = np.dot(xdata, self.__W2) + self.__b2
        a2 = sigmoid(z2)
        z3 = np.dot(a2, self.__W3) + self.__b3
        y = sigmoid(z3)
        return y[0], (1 if y[0] > 0.5 else 0)

np.random.seed(0)
xdata = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
xor = LogicGate('XOR', xdata, np.array([0, 1, 1, 0]))
xor.train()                                          # 10초 정도 걸립니다
for x in xdata:
    y, r = xor.predict(x)
    print(f'{x.tolist()} → y = {y:.4f} → {r}')

> **[확인 1]** 딥러닝 버전의 XOR 예측은 정답 (0, 1, 1, 0) 과 ( 일치한다 / 다르다 ).
> 제 9편(로지스틱 회귀 하나)의 손실 값 약 2.77 과 비교하면 지금 손실 값은 ( 　　 ) 이다.

💡 NAND·OR·AND 를 사람이 조합해 주지 않았는데도, 은닉층의 노드들이 **스스로 알맞은 역할** 을 학습했습니다!

## 🎬 애니메이션 2. 신경망이 만드는 '휘어진' 결정 경계

로지스틱 회귀의 경계는 직선이었지만, 은닉층이 있는 신경망의 경계는 **휘어질 수 있습니다.** 학습 과정을 재생합니다.

In [ ]:
#@title 🎬 애니메이션 2 실행 (실습을 먼저 실행하세요)
G1, G2 = np.meshgrid(np.linspace(-0.5, 1.5, 70), np.linspace(-0.5, 1.5, 70))
grid = np.c_[G1.ravel(), G2.ravel()]
tx = [0, 1, 1, 0]
n = len(xor.history)
sel = list(range(0, 30)) + list(range(30, n, 3)) + [n - 1]*6
fig, ax = plt.subplots(figsize=(6, 6))
def draw(f):
    st, W2k, b2k, W3k, b3k = xor.history[sel[f]]
    P = sigmoid(sigmoid(grid @ W2k + b2k) @ W3k + b3k).reshape(G1.shape)
    ax.clear()
    ax.contourf(G1, G2, P, levels=np.linspace(0, 1, 11), cmap='coolwarm', alpha=0.6)
    if P.min() < 0.5 < P.max():
        ax.contour(G1, G2, P, levels=[0.5], colors='k', linewidths=2.5)
    for (a, b_), tt in zip(xdata, tx):
        ax.scatter(a, b_, s=200, marker='o' if tt else 'x', color='darkred' if tt else 'navy', lw=3, zorder=5)
    ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5); ax.set_aspect('equal')
    ax.set_title(f'딥러닝 XOR  (step {st})', fontsize=14)
anim = animation.FuncAnimation(fig, draw, frames=len(sel), interval=120)
plt.close(fig)
show_anim(anim)

> **[관찰 1]** 학습 초반 한동안은 배경이 한 가지 색이다가, step ( 　　 ) 쯤부터 경계가 생긴다. 완성된 경계는 ● 두 개를 ( 하나의 띠 / 두 영역 ) 로 묶는다.

## ✏️ 연습 문제

> **[문제 1]** 입력 2개, 은닉층 노드 10개, 출력 1개인 신경망에서 W2, b2, W3, b3 의 형상과 학습할 값의 총 개수를 구하세요.
> W2 ( 　 × 　 ), b2 ( 　 ), W3 ( 　 × 　 ), b3 ( 　 ) → 총 ( 　 ) 개

> **[문제 2]** 실습의 피드포워드를 행렬 식으로 직접 써서, 학습이 끝난 XOR 신경망으로 입력 (1, 0) 의 출력을 계산하는 코드를 완성하세요.

In [ ]:
st, W2k, b2k, W3k, b3k = xor.history[-1]            # 학습이 끝난 가중치
A1 = np.array([1, 0])
A2 = sigmoid(np.dot(A1, ______) + ______)
A3 = sigmoid(np.dot(______, W3k) + b3k)
print('y =', A3)

In [ ]:
#@title 🔑 정답 확인 { display-mode: "form" }
print('문제 1 : W2 (2 × 10) = 20개, b2 = 10개, W3 (10 × 1) = 10개, b3 = 1개 → 총 41개')
st, W2k, b2k, W3k, b3k = xor.history[-1]
A1 = np.array([1, 0]); A2 = sigmoid(np.dot(A1, W2k) + b2k); A3 = sigmoid(np.dot(A2, W3k) + b3k)
print('문제 2 : A2 = sigmoid(A1·W2 + b2), A3 = sigmoid(A2·W3 + b3) →  y =', A3)

## 📌 정리

| 핵심 | 내용 |
|:---|:---|
| 노드 | 로지스틱 회귀 1개 (선형 회귀 + 활성화 함수) |
| 구조 | 입력층 → 은닉층(1개 이상) → 출력층 |
| 피드포워드 | A<sup>(2)</sup> = sigmoid(A<sup>(1)</sup> · W<sup>(2)</sup> + b<sup>(2)</sup>),  A<sup>(3)</sup> = sigmoid(A<sup>(2)</sup> · W<sup>(3)</sup> + b<sup>(3)</sup>) |
| 학습 | 출력층의 손실을 줄이도록 **모든 층의** W, b 를 경사하강법으로 갱신 |
| 의미 | 은닉층 덕분에 직선으로 나눌 수 없는 문제(XOR)도 풀 수 있다 |

### 🏆 도전 과제
XOR 은 입력이 2개였습니다. 28 × 28 손글씨 숫자 사진은 입력이 ( 　 ) 개, 정답은 0 ~ 9 중 하나이므로 출력 노드는 ( 　 ) 개가 필요합니다.
→ 다음 시간: **MNIST 손글씨 숫자 인식**

---

[⬅ 이전 : 제 9편 XOR 문제 - 직선 하나로는 나눌 수 없다](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/09_xor.ipynb)　|　[📚 목차](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/00_index.ipynb)　|　[다음 : 제 11편 MNIST - 손글씨 숫자를 읽는 신경망 ➡](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/11_mnist.ipynb)